In [1]:
import pandas as pd

In [2]:
def check_lemmas(lemmas, min_char=2):
    lemmas = lemmas.split()
    for lemma in lemmas:
        if ((len(lemma) < min_char) or
            (not lemma.isalpha())):
            out = False
            break
    else:
        out = True
    return out

In [3]:
# Load data
df = pd.read_parquet('cacl_t1_onesentperline_lemmatized_lower_5grams_freqs_inner_ll.parquet')
df['lemmas_clean'] = df['lemma'].apply(check_lemmas)

gpt_df = pd.read_parquet('sample_gpt35_improved_input_lemmatized_lower_5grams_freqs_inner_ll.parquet')
gpt_df['lemmas_clean'] = gpt_df['lemma'].apply(check_lemmas)

In [4]:
# Retrieve top 40 from naturalistic corpus

df_hi = df[(df['ratio_t2']>1) & 
           (df['lemmas_clean']) &
           (df['freq2']>=10) &
           (df['freq1']>=10)].sort_values('ll', ascending=False)[:40]

df_lo = df[(df['ratio_t2']<1) & 
           (df['lemmas_clean']) &
           (df['freq2']>=10) &
           (df['freq1']>=10)].sort_values('ll', ascending=False)[:40]

In [5]:
# Retrieve top 40 from LLM corpus (direct comparison)

gpt_df_hi = gpt_df[(gpt_df['ratio_t2']>1) &
                   (gpt_df['lemmas_clean'])].sort_values('ll', ascending=False)[:40]

gpt_df_lo = gpt_df[(gpt_df['ratio_t2']<1) &
                   (gpt_df['lemmas_clean'])].sort_values('ll', ascending=False)[:40]

In [6]:
# Retrieve top 400 from LLM corpus (only overlap check)

gpt_df_hi400 = gpt_df[(gpt_df['ratio_t2']>1) &
                      (gpt_df['lemmas_clean'])].sort_values('ll', ascending=False)[:400]['target'].to_list()

gpt_df_lo400 = gpt_df[(gpt_df['ratio_t2']<1) &
                      (gpt_df['lemmas_clean'])].sort_values('ll', ascending=False)[:400]['target'].to_list()

gpt_df_hi400 = set(gpt_df_hi400)
gpt_df_lo400 = set(gpt_df_lo400)

In [7]:
# Check overlap
df_hi['in_gpt'] = df['target'].apply(lambda x: '{*}' if x in gpt_df_hi400 else '   ')
df_lo['in_gpt'] = df['target'].apply(lambda x: '{*}' if x in gpt_df_lo400 else '   ')

In [8]:
# Prepare output
out_cols = ['lemma', 'freq_norm1', 'freq_norm2', 'll']

out_df_hi = pd.concat([df_hi[['in_gpt'] + out_cols].reset_index(drop=True), 
                       gpt_df_hi[out_cols].reset_index(drop=True)], 
                      axis=1)

out_df_lo = pd.concat([df_lo[['in_gpt'] + out_cols].reset_index(drop=True), 
                       gpt_df_lo[out_cols].reset_index(drop=True)], 
                      axis=1)

out_df = pd.concat([out_df_hi, out_df_lo]).reset_index(drop=True)

In [9]:
out_df.round(1).head()

,in_gpt,lemma,freq_norm1,freq_norm2,ll,lemma,freq_norm1,freq_norm2,ll
0,,consistent with their intend use,0.1,18.5,2440.5,be important to note that,8.2,72.1,28.6
1,,be consistent with their intend,0.1,18.3,2410.8,it be important to note,10.2,72.1,25.6
2,,identify individual people or offensive,0.1,17.8,2369.2,important to note that the,2.0,24.7,11.1
3,,information that name or uniquely,0.1,17.7,2365.9,it be worth note that,8.2,39.1,10.8
4,,that name or uniquely identify,0.1,17.7,2365.9,can be find in appendix,2.0,20.6,8.6


In [10]:
# Print LaTeX version (uncomment)
# print(out_df.to_latex(float_format='%.1f', index=False, escape=False))